# DBMS Phase 1 — Fundamentals and Architecture

**Roadmap source:** `dbms_complete_roadmap.md` → Phase 1: Database Fundamentals & Architecture (Lecture 1).

This notebook is exam-oriented and software-engineering oriented. For every topic, follow this order:

1. **Learn:** read the definition, mechanism, and exam language.
2. **Observe:** run the small demonstration.
3. **Implement:** complete the `YOUR TURN` cell without copying a solution.
4. **Explain:** answer the oral-exam questions in your own words.

## Phase 1 completion bar

You should be able to explain why a DBMS exists, trace a query through its engine, distinguish the three schema levels, defend physical versus logical data independence, and choose between an OLTP and an OLAP design.

## 0. The central exam idea

A database is not merely a file containing rows. A **DBMS** is software that provides a controlled, shared, reliable way to store and use data. It manages the boundary between applications and physical storage.

A useful one-sentence answer is:

> A DBMS provides data abstraction, declarative querying, integrity enforcement, transaction management, concurrency control, recovery, and security over shared persistent data.

Keep this sentence in mind: every Phase 1 topic explains one part of this promise.

# Part I — The seven flaws of traditional file-processing systems

A file-processing system stores data in application-owned files such as CSV, JSON, binary files, or spreadsheets. It can work for a small isolated script, but it becomes dangerous when multiple applications and users share the same facts.

For exams, do not only memorize the names. Be able to state: **the failure mechanism, a concrete example, and the DBMS capability that addresses it.**

| Flaw | What happens | Example | DBMS response |
|---|---|---|---|
| Data redundancy and inconsistency | The same fact is copied and updates diverge | HR and payroll store different addresses | Central schema and single source of truth |
| Difficulty accessing data | Every new question needs custom parsing code | Write a new script for monthly revenue | Declarative SQL and query processing |
| Data isolation | Data is split across incompatible files | Joining CSV, JSON, and binary records | Integrated relational model and joins |
| Integrity problems | Rules live in application code and can be bypassed | Negative account balance enters a file | Central constraints such as `CHECK`, keys, and foreign keys |
| Atomicity failures | A crash leaves only part of a multi-step operation | Debit succeeds but credit fails | Transactions and rollback/recovery |
| Concurrent-access anomalies | Simultaneous writers overwrite or observe unsafe states | Lost update on an inventory count | Concurrency control and isolation |
| Security and access-control weakness | File permissions are too coarse | A reporting script can read salary columns | Users, privileges, views, and row/column controls |

## 1. Redundancy, inconsistency, isolation, and integrity — observe

The following simulates two department-owned records. The program has no shared constraint or single update path, so a fact can diverge. It also has to write its own filtering and joining logic.

In [1]:
hr_file = [
    {'employee_id': 'E-17', 'name': 'Mira', 'address': '12 Lake Road'},
]
payroll_file = [
    {'employee_id': 'E-17', 'salary': 85000, 'address': '12 Lake Rd'},
]

# The same employee has two spellings for one fact. Nothing enforces consistency.
print(hr_file[0]['address'])
print(payroll_file[0]['address'])

# A custom application join is needed to combine the files.
joined_record = {**hr_file[0], **payroll_file[0]}
print(joined_record)

12 Lake Road
12 Lake Rd
{'employee_id': 'E-17', 'name': 'Mira', 'address': '12 Lake Rd', 'salary': 85000}


### Learn → implement: centralize an integrity rule

Application validation is useful, but it is not sufficient: another script can bypass it. A DBMS lets the storage layer enforce rules for every writer.

Implement the function below as a small demonstration of a centralized domain rule. It must accept only a non-empty employee ID and a non-negative salary. Raise `ValueError` for invalid input.

In [2]:
# YOUR TURN — completed example
def validate_employee_record(employee_id: str, salary: int) -> dict[str, str | int]:
    cleaned_employee_id = employee_id.strip()
    if not cleaned_employee_id:
        raise ValueError('employee_id must not be empty')
    if salary < 0:
        raise ValueError('salary must not be negative')
    return {'employee_id': cleaned_employee_id, 'salary': salary}


In [3]:
# Checks — run after implementing the validator.
assert validate_employee_record('E-17', 85000) == {
    'employee_id': 'E-17',
    'salary': 85000,
}
for invalid_record in (('', 85000), ('E-17', -1)):
    try:
        validate_employee_record(*invalid_record)
    except ValueError:
        pass
    else:
        raise AssertionError(f'{invalid_record} should be rejected')
print('Integrity checks passed.')

Integrity checks passed.


## 2. Atomicity and concurrent-access anomalies — observe

**Atomicity** means a logical operation is all-or-nothing. A bank transfer is not complete if only the debit is stored.

A **lost update** happens when two transactions read the same old value, compute independent new values, and the later write overwrites the earlier write. The final state silently loses one user's work. A DBMS uses locks, MVCC, or another concurrency-control mechanism to prevent unsafe interleavings.

In [4]:
# Atomicity failure in a naive file-style program.
accounts = {'A': 100, 'B': 50}
accounts['A'] -= 30          # debit written
# Imagine a crash here before the credit is written.
print(accounts)                # money disappeared from the system

# Lost-update interleaving. Both workers read 10 before either writes.
stock = {'SKU-1': 10}
worker_a_reads = stock['SKU-1']
worker_b_reads = stock['SKU-1']
stock['SKU-1'] = worker_a_reads - 1
stock['SKU-1'] = worker_b_reads - 1
print(stock['SKU-1'])  # 9, even though two reservations occurred

{'A': 70, 'B': 50}
9


## 3. Security weakness — observe

Operating-system file permissions often answer only “can this process read the file?” A DBMS can answer finer questions: may this role read only a view, update only certain rows, or access a column at all?

Treat this as a design principle: least privilege means giving each application only the data and operations it needs.

In [5]:
employee_records = [
    {'employee_id': 'E-17', 'name': 'Mira', 'salary': 85000},
    {'employee_id': 'E-18', 'name': 'Arun', 'salary': 72000},
]

# A careless file reader returns every field. A DBMS view can expose only approved columns.
public_directory = [
    {'employee_id': row['employee_id'], 'name': row['name']}
    for row in employee_records
]
print(public_directory)

[{'employee_id': 'E-17', 'name': 'Mira'}, {'employee_id': 'E-18', 'name': 'Arun'}]


### Exam answer pattern for the seven flaws

If asked “why not just use files?”, answer with a chain rather than a list: duplicated facts become inconsistent; isolated formats make access and joins application-specific; rules are bypassable; crashes break multi-step work; concurrent writers interfere; and coarse file permissions expose too much. A DBMS centralizes these concerns behind a controlled data model and execution engine.

# Part II — DBMS abstraction and components

## Core services

A practical DBMS provides:

- a centralized data model and catalog;
- declarative query language support;
- domain, entity, and referential integrity enforcement;
- transaction and concurrency management;
- recovery after failures; and
- authentication, authorization, and auditing.

The application says **what** data it wants. The DBMS chooses **how** to obtain and protect it.

## Engine pipeline

```text
SQL text
  ↓
Query parser → parsed/validated representation
  ↓
Query optimizer → cheaper logical/physical plan
  ↓
Execution engine → operators run the plan
  ↔
Access methods → heap/table scan or B+Tree index lookup
  ↔
Buffer pool → caches database pages in memory
  ↔
Storage manager + WAL → durable pages and crash recovery
```

Important distinctions:

- **Parser:** checks grammar and resolves names; it does not execute the query.
- **Optimizer:** compares possible plans using statistics and estimated cost.
- **Execution engine:** runs operators such as scan, filter, join, and aggregate.
- **Access method:** decides how records are located; a heap scan reads pages, while a B+Tree can locate indexed keys.
- **Buffer pool:** avoids repeated disk I/O by caching pages; dirty pages must eventually be flushed safely.
- **WAL (Write-Ahead Logging):** log records describing changes reach durable storage before the corresponding data pages, allowing REDO/UNDO-style recovery.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class QueryRequest:
    table: str
    selected_columns: tuple[str, ...]
    minimum_salary: int


def parse_query(request: QueryRequest) -> QueryRequest:
    # A real parser handles SQL grammar; this toy parser validates a request object.
    if not request.table or not request.selected_columns:
        raise ValueError('table and selected_columns are required')
    return request


def choose_plan(request: QueryRequest, has_salary_index: bool) -> str:
    # A real optimizer uses statistics and many cost estimates.
    return 'B+Tree index scan' if has_salary_index else 'heap/table scan'


def execute_plan(
    request: QueryRequest,
    rows: list[dict[str, object]],
) -> list[dict[str, object]]:
    return [
        {column: row[column] for column in request.selected_columns}
        for row in rows
        if row['salary'] >= request.minimum_salary
    ]


request = parse_query(QueryRequest('employees', ('employee_id', 'name'), 80000))
rows = [
    {'employee_id': 'E-17', 'name': 'Mira', 'salary': 85000},
    {'employee_id': 'E-18', 'name': 'Arun', 'salary': 72000},
]
print('Plan:', choose_plan(request, has_salary_index=False))
print('Result:', execute_plan(request, rows))

Plan: heap/table scan
Result: [{'employee_id': 'E-17', 'name': 'Mira'}]


### Learn → implement: a miniature execution pipeline

Complete the function below. It represents the execution-engine part only: filter rows by `minimum_salary` and project the requested columns. The input data is already in memory, so do not implement SQL parsing or an actual index.

Production reasoning: the function should not mutate `rows`; it should return new result dictionaries.

In [ ]:
# YOUR TURN
def execute_salary_query(
    rows: list[dict[str, object]],
    selected_columns: tuple[str, ...],
    minimum_salary: int,
) -> list[dict[str, object]]:
    # Filter, then project. Preserve input rows.
    return [
        {column: row[column] for column in selected_columns}
        for row in rows
        if row['salary'] >= minimum_salary
    ]


In [ ]:
# Checks — run after implementing execute_salary_query.
source_rows = [
    {'employee_id': 'E-17', 'name': 'Mira', 'salary': 85000},
    {'employee_id': 'E-18', 'name': 'Arun', 'salary': 72000},
]
expected = [{'employee_id': 'E-17', 'name': 'Mira'}]
assert execute_salary_query(source_rows, ('employee_id', 'name'), 80000) == expected
assert source_rows[0]['salary'] == 85000
print('Execution-engine checks passed.')

Execution-engine checks passed.


## Storage concepts you must be able to define

**Heap:** an unordered collection of table records/pages; a table scan may inspect many pages.

**B+Tree:** a balanced tree index whose leaves are ordered and linked; it supports efficient equality and range lookups when the predicate matches the index.

**Buffer pool:** the in-memory page cache between the execution engine and storage. A cache hit avoids disk I/O.

**Storage manager:** allocates pages, reads/writes records, manages files and indexes, and coordinates durable persistence.

**WAL:** before a changed page is flushed, its log record must be durable. After a crash, the DBMS can use the log to recover committed work and undo or ignore incomplete work.

Exam trap: an index is not the table itself. It is an access path that trades storage and write-maintenance cost for faster eligible lookups.

# Part III — Three-schema architecture and data independence

The three-schema architecture separates user-facing views from the logical database design and the physical storage details. It reduces the number of application programs that must change when one layer changes.

```text
External level   → user/application views
       ↓
Conceptual level → global logical schema: entities, relationships, constraints
       ↓
Internal level   → files, pages, record layout, indexes, compression
```

### The two independence guarantees

- **Physical data independence:** change internal storage—add/drop an index, change page layout, compress data—without changing the conceptual schema or application queries.
- **Logical data independence:** change the conceptual schema—split a table, add an attribute, introduce a relationship—without breaking external views or applications, provided the view contract is preserved.

Physical independence is generally easier to achieve than logical independence because applications depend heavily on logical names and meanings.

In [ ]:
import sqlite3

connection = sqlite3.connect(':memory:')
connection.execute("CREATE TABLE employees (employee_id TEXT PRIMARY KEY, name TEXT NOT NULL, salary INTEGER NOT NULL)")
connection.executemany(
    'INSERT INTO employees(employee_id, name, salary) VALUES (?, ?, ?)',
    [('E-17', 'Mira', 85000), ('E-18', 'Arun', 72000)],
)

# Conceptual/base table: the logical employee structure.
# External level: expose only the columns a directory application needs.
connection.execute("CREATE VIEW public_employee_directory AS SELECT employee_id, name FROM employees")
print(connection.execute('SELECT * FROM public_employee_directory').fetchall())

# Internal change: add an index. The view query and its result contract stay the same.
connection.execute('CREATE INDEX employees_salary_idx ON employees(salary)')
print(connection.execute('SELECT * FROM public_employee_directory').fetchall())

[('E-17', 'Mira'), ('E-18', 'Arun')]
[('E-17', 'Mira'), ('E-18', 'Arun')]


### Learn → implement: identify the independence type

Classify each change as **physical independence**, **logical independence**, or **not independence**:

1. Add a B+Tree index on `employees.salary`; existing SQL remains unchanged.
2. Rename `employees.name` to `employees.full_name` and update every application query.
3. Split one logical `employees` table into `employees` and `employee_addresses`, while preserving an external view with the old columns.
4. Change a public view so that `salary` is exposed to a role that previously could not read it.

Write your answers and one-sentence justification in the next cell.

In [ ]:
# YOUR TURN — write answers such as: 1. physical independence — ...
answers = {
    1: 'Physical independence — adding an index changes internal access paths, not the conceptual schema or application queries.',
    2: 'Not independence — renaming a logical attribute breaks application queries that depend on the old name.',
    3: 'Logical independence — the conceptual tables can be split while an external view preserves the old application-facing contract.',
    4: 'Not independence — changing a public view changes the external/security contract and exposes data to a role that previously could not read it.',
}
print(answers)

{1: 'Physical independence — adding an index changes internal access paths, not the conceptual schema or application queries.', 2: 'Not independence — renaming a logical attribute breaks application queries that depend on the old name.', 3: 'Logical independence — the conceptual tables can be split while an external view preserves the old application-facing contract.', 4: 'Not independence — changing a public view changes the external/security contract and exposes data to a role that previously could not read it.'}


# Part IV — Workload models: OLTP versus OLAP

| Dimension | OLTP | OLAP |
|---|---|---|
| Goal | Run the business correctly in real time | Analyze history for decisions |
| Query shape | Short, selective reads/writes | Long scans, joins, and aggregations |
| Concurrency | Many concurrent users and transactions | Fewer analytical jobs, often large scans |
| Data layout | Row-oriented is often useful for whole-record access | Column-oriented is often useful for reading a few columns across many rows |
| Correctness priority | Low latency, constraints, ACID | Throughput, scan efficiency, analytical flexibility |
| Example | Order placement, account transfer, inventory reservation | Monthly revenue trend, cohort analysis, dashboards |

A system can contain both patterns, but the workload shapes influence indexes, partitioning, schema design, caching, and hardware. PostgreSQL is a common OLTP database; DuckDB is an example of an analytical engine designed for OLAP-style work. These are workload examples, not absolute labels.

In [16]:
# Same data, two different workload shapes.
orders = [
    {'order_id': 1, 'customer_id': 'C1', 'amount': 25.0},
    {'order_id': 2, 'customer_id': 'C2', 'amount': 80.0},
    {'order_id': 3, 'customer_id': 'C1', 'amount': 40.0},
]

# OLTP-shaped: retrieve or update one known order.
order_for_update = next(order for order in orders if order['order_id'] == 2)
order_for_update['amount'] = 85.0
print(order_for_update)

# OLAP-shaped: scan all orders and compute a grouped summary.
revenue_by_customer: dict[str, float] = {}
for order in orders:
    customer_id = order['customer_id']
    revenue_by_customer[customer_id] = revenue_by_customer.get(customer_id, 0.0) + order['amount']
print(revenue_by_customer)

{'order_id': 2, 'customer_id': 'C2', 'amount': 85.0}
{'C1': 65.0, 'C2': 85.0}


### Learn → implement: classify the workload

Implement `classify_workload(operation: str) -> str`. Return `'OLTP'` for operations that are short, concurrent, and part of live business state; return `'OLAP'` for historical scans, aggregations, and reporting. Raise `ValueError` for an unknown operation.

Use an explicit mapping or a small set of known operation names. Do not infer from arbitrary text yet.

In [ ]:
# YOUR TURN
def classify_workload(operation: str) -> str:
    oltp_operations = {'place_order', 'reserve_inventory'}
    olap_operations = {'monthly_revenue_report', 'customer_cohort_analysis'}
    if operation in oltp_operations:
        return 'OLTP'
    if operation in olap_operations:
        return 'OLAP'
    raise ValueError(f'Unknown operation: {operation!r}')


In [ ]:
# Checks — run after implementing classify_workload.
assert classify_workload('place_order') == 'OLTP'
assert classify_workload('reserve_inventory') == 'OLTP'
assert classify_workload('monthly_revenue_report') == 'OLAP'
assert classify_workload('customer_cohort_analysis') == 'OLAP'
try:
    classify_workload('unknown_operation')
except ValueError:
    pass
else:
    raise AssertionError('Unknown operations must raise ValueError')
print('Workload-classification checks passed.')

# Exam revision sheet

## Short-answer questions

1. Explain all seven flaws of file-processing systems and give one DBMS capability for each.
2. Trace `SELECT ... WHERE ...` through parser, optimizer, execution engine, access method, buffer pool, and storage.
3. What is the difference between a heap scan and a B+Tree index lookup?
4. Why must WAL reach durable storage before a dirty data page is flushed?
5. Define external, conceptual, and internal schema levels.
6. Contrast physical and logical data independence with one example each.
7. Why is logical data independence usually harder?
8. Contrast OLTP and OLAP in query shape, concurrency, and data layout.

## Oral-exam answer template

For any architecture component, answer in this order: **definition → job → what problem it solves → concrete example → trade-off or limitation**.

## Phase 1 checkpoint

Do not move to Phase 2 until you can draw the engine pipeline from memory, explain the three schema levels without mixing them up, and distinguish a lost update from an atomicity failure. Complete the `YOUR TURN` cells first; then send me your answers or one cell at a time for review.